# 02 - Data Preprocessing
## Milk Quality Prediction — Feature Engineering & Scaling

**Tujuan:** Mengaplikasikan pipeline preprocessing (`FeatureEngineer` + `StandardScaler`) dari modul `api.ml.pipeline`, membagi data train/test secara stratified, dan menyimpan hasil preprocessing untuk modeling.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'api'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

from train_model import generate_synthetic_data
from ml.pipeline import (
    FEATURE_COLUMNS, NUMERIC_FEATURES, BINARY_FEATURES, ORDINAL_FEATURES,
    FeatureEngineer, create_pipeline, GRADE_MAPPING, INVERSE_GRADE_MAPPING
)

print('Libraries and API modules loaded successfully.')

---
## 1. Load Data

In [ ]:
df = generate_synthetic_data(n=3000)
print(f'Shape: {df.shape}')
df.head(3)

---
## 2. Pipeline Definition

Pipeline yang didefinisikan di `api.ml.pipeline` terdiri dari dua langkah:

1. **`FeatureEngineer`** — Custom transformer yang:
   - Mengisi missing value numeric dengan median.
   - Mengisi missing value biner/ordinal dengan 0.
   - Menambahkan kolom yang hilang dengan nilai 0.
   - Mengembalikan DataFrame dengan urutan kolom sesuai `FEATURE_COLUMNS`.

2. **`StandardScaler`** — Standardisasi (mean=0, std=1) untuk semua fitur.

In [ ]:
pipeline = create_pipeline()
print(pipeline)

---
## 3. Fit & Transform Pipeline

In [ ]:
X = df.drop(columns=['grade'])
y = df['grade'].map(GRADE_MAPPING)

X_processed = pipeline.fit_transform(X)

print(f'X shape (raw):      {X.shape}')
print(f'X shape (processed): {X_processed.shape}')
print(f'y shape:             {y.shape}')
print(f'\nColumn order after pipeline:')
print(list(X_processed.columns) if hasattr(X_processed, 'columns') else 'Numpy array')

## 4. Before vs After Scaling — Comparison

In [ ]:
X_before = X[FEATURE_COLUMNS]
X_after = pd.DataFrame(X_processed, columns=FEATURE_COLUMNS)

stats_before = X_before.describe().T[['mean', 'std', 'min', 'max']]
stats_after = X_after.describe().T[['mean', 'std', 'min', 'max']]

print('=== Before Scaling (Raw) ===')
display(stats_before.round(4))

print('\n=== After Scaling (Standardized) ===')
display(stats_after.round(4))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sample_features = ['ph', 'fat', 'added_water', 'temperature']

for idx, feat in enumerate(sample_features):
    ax = axes[idx // 2, idx % 2]
    ax.hist(X_before[feat], bins=30, alpha=0.6, label='Before', color='#3498db', edgecolor='white')
    ax.hist(X_after[feat], bins=30, alpha=0.6, label='After (scaled)', color='#e74c3c', edgecolor='white')
    ax.set_title(f'{feat} — Before vs After Scaling', fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.legend()

plt.tight_layout()
plt.show()

**Observasi:** Setelah scaling, seluruh fitur memiliki mean ≈ 0 dan std ≈ 1. Rentang nilai menjadi seragam, yang sangat membantu model berbasis jarak (Logistic Regression) maupun tree-based (Random Forest/XGBoost) dalam konvergensi.

---
## 5. Train / Test Split (Stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape:  {y_test.shape}')

print('\nGrade distribution in train set:')
train_dist = y_train.value_counts().sort_index()
for g in range(4):
    label = INVERSE_GRADE_MAPPING[g]
    print(f'  {label}: {train_dist[g]} ({train_dist[g]/len(y_train)*100:.1f}%)')

print('\nGrade distribution in test set:')
test_dist = y_test.value_counts().sort_index()
for g in range(4):
    label = INVERSE_GRADE_MAPPING[g]
    print(f'  {label}: {test_dist[g]} ({test_dist[g]/len(y_test)*100:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

train_counts = y_train.map(INVERSE_GRADE_MAPPING).value_counts().reindex(['A', 'B', 'C', 'Reject'])
test_counts = y_test.map(INVERSE_GRADE_MAPPING).value_counts().reindex(['A', 'B', 'C', 'Reject'])

colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']

train_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_title('Train Set — Grade Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

test_counts.plot(kind='bar', ax=axes[1], color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_title('Test Set — Grade Distribution', fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

Dengan `stratify=y`, distribusi grade di train dan test set proporsional terhadap distribusi asli.

---
## 6. Save Preprocessed Data

In [ ]:
save_dir = os.path.join(os.getcwd(), '..', 'data', 'processed')
os.makedirs(save_dir, exist_ok=True)

np.save(os.path.join(save_dir, 'X_train.npy'), X_train)
np.save(os.path.join(save_dir, 'X_test.npy'), X_test)
np.save(os.path.join(save_dir, 'y_train.npy'), y_train)
np.save(os.path.join(save_dir, 'y_test.npy'), y_test)

train_df = pd.DataFrame(X_train, columns=FEATURE_COLUMNS)
train_df['grade'] = y_train.map(INVERSE_GRADE_MAPPING)
train_df.to_csv(os.path.join(save_dir, 'train.csv'), index=False)

test_df = pd.DataFrame(X_test, columns=FEATURE_COLUMNS)
test_df['grade'] = y_test.map(INVERSE_GRADE_MAPPING)
test_df.to_csv(os.path.join(save_dir, 'test.csv'), index=False)

print(f'Preprocessed data saved to: {save_dir}')
print(f'  X_train.npy  — {X_train.shape}')
print(f'  X_test.npy   — {X_test.shape}')
print(f'  y_train.npy  — {y_train.shape}')
print(f'  y_test.npy   — {y_test.shape}')
print(f'  train.csv')
print(f'  test.csv')

---
## 7. Summary

| Step | Detail |
|------|--------|
| **FeatureEngineer** | Imputasi median untuk numeric, 0 untuk biner/ordinal; menjamin semua FEATURE_COLUMNS ada |
| **StandardScaler** | Standardisasi (mean=0, std=1) — penting untuk convergence model linear |
| **Train/Test Split** | 80/20 dengan stratifikasi — distribusi grade terjaga |
| **Persisted** | Data disimpan ke `data/processed/` sebagai `.npy` dan `.csv` untuk modeling |

Pipeline siap digunakan untuk training **Baseline Models** (notebook 03).